# 05 - CNN Training in Google Colab

This notebook trains the CNN in **Google Colab**.

It does not need the raw SD302 folders. It loads the preprocessed package created locally by:

`04_cnn_preprocessing_for_colab.ipynb`

The package should be stored in your private Google Drive.

## Important data note

The package used here is derived from biometric fingerprint data.

Keep it private in your own Google Drive and use it only for this research/training project.

## Section 1: Install and import libraries

Colab usually has most of these installed, but this makes the notebook reproducible.

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn torch torchvision tqdm

In [ ]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## Section 2: Mount Google Drive

Run this cell in Colab and approve Drive access.

The training package should already be uploaded to Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Section 3: Set the package path

Update this path if you saved the zip file somewhere else in Google Drive.

In [ ]:
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/dermatoglyphic_project")
PACKAGE_PATH = DRIVE_PROJECT_DIR / "roll_cnn_colab_package.zip"

WORK_DIR = Path("/content/dermatoglyphic_project")
DATA_DIR = WORK_DIR / "data"
MODEL_DIR = DRIVE_PROJECT_DIR / "models"
FIGURE_DIR = DRIVE_PROJECT_DIR / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PACKAGE_PATH

In [ ]:
if not PACKAGE_PATH.exists():
    raise FileNotFoundError(f"Package not found: {PACKAGE_PATH}")

print("Package found:", PACKAGE_PATH)

## Section 4: Unzip and load the prepared data

The zip file contains the preprocessed image arrays, labels, split information, and metadata.

In [ ]:
required_files = {
    "roll_cnn_160x160_dataset.npz",
    "roll_cnn_metadata.csv",
    "label_mapping.json",
    "README_colab_package.txt",
}

with zipfile.ZipFile(PACKAGE_PATH, "r") as zip_file:
    corrupt_file = zip_file.testzip()
    if corrupt_file is not None:
        raise RuntimeError(f"Corrupt file in package: {corrupt_file}")

    missing_files = required_files.difference(zip_file.namelist())
    if missing_files:
        raise FileNotFoundError(f"Files missing from package: {sorted(missing_files)}")

    zip_file.extractall(DATA_DIR)

print("Package integrity check passed.")
sorted(path.name for path in DATA_DIR.iterdir())

In [ ]:
prepared_data = np.load(DATA_DIR / "roll_cnn_160x160_dataset.npz", allow_pickle=True)
metadata = pd.read_csv(DATA_DIR / "roll_cnn_metadata.csv")

images = prepared_data["images"]
labels = prepared_data["labels"]
splits = prepared_data["splits"]
label_names = prepared_data["label_names"].tolist()

expected_rows = 2312
expected_split_counts = {"train": 1620, "validation": 344, "test": 348}

if images.shape != (expected_rows, 160, 160):
    raise ValueError(f"Unexpected image shape: {images.shape}")
if len(labels) != expected_rows or len(splits) != expected_rows or len(metadata) != expected_rows:
    raise ValueError("Images, labels, splits, and metadata do not have matching row counts.")
if metadata.isna().any().any():
    raise ValueError("The metadata contains missing values.")
if not np.array_equal(labels, metadata["label_id"].to_numpy(dtype=np.int64)):
    raise ValueError("Saved labels do not match the metadata.")
if not np.array_equal(splits, metadata["split"].to_numpy()):
    raise ValueError("Saved splits do not match the metadata.")

actual_split_counts = pd.Series(splits).value_counts().to_dict()
if actual_split_counts != expected_split_counts:
    raise ValueError(f"Unexpected split counts: {actual_split_counts}")

print("Full dataset validation passed.")
print("Images:", images.shape)
print("Labels:", labels.shape)
print("Splits:", actual_split_counts)
print("Classes:", label_names)

In [ ]:
pd.crosstab(metadata["broad_class"], metadata["split"])[["train", "validation", "test"]]

## Section 5: Create PyTorch datasets

The data is already resized, so this dataset reads image arrays directly from memory.

Training uses light augmentation. Validation and test data are not augmented.

In [ ]:
IMAGE_SIZE = 160
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomRotation(8),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_transform = transforms.Compose([
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

In [ ]:
class FingerprintArrayDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0) / 255.0
        label = int(self.labels[index])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [ ]:
train_mask = splits == "train"
validation_mask = splits == "validation"
test_mask = splits == "test"

train_dataset = FingerprintArrayDataset(images[train_mask], labels[train_mask], train_transform)
validation_dataset = FingerprintArrayDataset(images[validation_mask], labels[validation_mask], eval_transform)
test_dataset = FingerprintArrayDataset(images[test_mask], labels[test_mask], eval_transform)

len(train_dataset), len(validation_dataset), len(test_dataset)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Section 6: Preview one training batch

This confirms that the arrays loaded correctly before training.

In [ ]:
batch_images, batch_labels = next(iter(train_loader))

print("Batch images:", batch_images.shape)
print("Batch labels:", batch_labels.shape)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(10, 7))

for axis, image, label in zip(axes.ravel(), batch_images[:12], batch_labels[:12]):
    axis.imshow(image.squeeze(), cmap="gray")
    axis.set_title(label_names[int(label)])
    axis.axis("off")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "colab_cnn_batch_preview.png", dpi=150)
plt.show()

## Stop and review Colab data loading

Pause here before training.

Check that:

- Google Drive mounted successfully.
- the package was found and unzipped.
- image shape is `(2312, 160, 160)`.
- the batch preview shows fingerprint images.

## Section 7: Define the CNN model

This is a compact CNN for the first Colab training run.

In [ ]:
class FingerprintCNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(128, number_of_classes),
        )

    def forward(self, images):
        features = self.features(images)
        return self.classifier(features)

In [ ]:
model = FingerprintCNN(number_of_classes=len(label_names)).to(device)
model

## Section 8: Class weights for imbalance

The `arch` class is smaller, so class weights help the model treat its mistakes as more important.

In [ ]:
train_labels = labels[train_mask]
train_counts = pd.Series(train_labels).value_counts().sort_index()

class_weights = len(train_labels) / (len(label_names) * train_counts)
class_weights = torch.tensor(class_weights.values, dtype=torch.float32).to(device)

pd.Series(class_weights.cpu().numpy(), index=label_names)

## Section 9: Train the CNN

The best model is selected by validation accuracy and saved to Google Drive.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
best_model_path = MODEL_DIR / "cnn_roll_colab_best.pt"

In [ ]:
def run_epoch(model, data_loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0
    total_correct = 0
    total_images = 0

    for images, labels_batch in data_loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels_batch).sum().item()
        total_images += images.size(0)

    return total_loss / total_images, total_correct / total_images

In [ ]:
history = []
best_validation_accuracy = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
    validation_loss, validation_accuracy = run_epoch(model, validation_loader, criterion)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
    })

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(model.state_dict(), best_model_path)

    print(f"Epoch {epoch:02d}: train_acc={train_accuracy:.3f}, val_acc={validation_accuracy:.3f}")

print("Best validation accuracy:", round(best_validation_accuracy, 4))

## Section 10: Plot training history

These plots show whether the CNN is learning and whether it is overfitting.

In [ ]:
history_table = pd.DataFrame(history)
history_path = DRIVE_PROJECT_DIR / "cnn_colab_training_history.csv"

history_table.to_csv(history_path, index=False)
history_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_table["epoch"], history_table["train_accuracy"], label="train")
axes[0].plot(history_table["epoch"], history_table["validation_accuracy"], label="validation")
axes[0].set_title("CNN Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history_table["epoch"], history_table["train_loss"], label="train")
axes[1].plot(history_table["epoch"], history_table["validation_loss"], label="validation")
axes[1].set_title("CNN Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURE_DIR / "cnn_colab_training_history.png", dpi=150)
plt.show()

## Stop and review CNN training

Pause here after training.

Check the validation accuracy and the training curves before running the final test evaluation.

## Section 11: Evaluate the best model on the test set

The test set is used once for final evaluation.

In [ ]:
best_model = FingerprintCNN(number_of_classes=len(label_names)).to(device)
best_model.load_state_dict(torch.load(best_model_path, map_location=device))
best_model.eval()

print("Loaded best model:", best_model_path)

In [ ]:
def collect_predictions(model, data_loader):
    all_predictions = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images_batch, labels_batch in data_loader:
            images_batch = images_batch.to(device)
            outputs = model(images_batch)

            all_predictions.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels_batch.numpy())

    return np.array(all_labels), np.array(all_predictions)

In [ ]:
test_labels, test_predictions = collect_predictions(best_model, test_loader)

test_accuracy = accuracy_score(test_labels, test_predictions)
print("CNN test accuracy:", round(test_accuracy, 4))

In [ ]:
cnn_report = classification_report(
    test_labels,
    test_predictions,
    target_names=label_names,
    output_dict=True,
)

cnn_report_table = pd.DataFrame(cnn_report).T
cnn_report_table.round(3)

## Section 12: Confusion matrix

The confusion matrix shows where the CNN succeeds and where it still struggles.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay.from_predictions(
    test_labels,
    test_predictions,
    display_labels=label_names,
    xticks_rotation=25,
    cmap="Blues",
    ax=axis,
)

axis.set_title("CNN Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cnn_colab_confusion_matrix.png", dpi=150)
plt.show()

## Section 13: Save test outputs

The report and predictions are saved back to Google Drive.

In [ ]:
report_path = DRIVE_PROJECT_DIR / "cnn_colab_test_classification_report.csv"
prediction_path = DRIVE_PROJECT_DIR / "cnn_colab_test_predictions.csv"

cnn_report_table.to_csv(report_path)

test_results = metadata[metadata["split"] == "test"].copy().reset_index(drop=True)
test_results["true_label"] = [label_names[index] for index in test_labels]
test_results["predicted_label"] = [label_names[index] for index in test_predictions]
test_results["correct"] = test_results["true_label"] == test_results["predicted_label"]
test_results.to_csv(prediction_path, index=False)

print("Saved:", report_path)
print("Saved:", prediction_path)

## Stop and discuss CNN results

Pause here after final evaluation.

Compare this CNN against the HOG baseline using accuracy, macro F1-score, per-class recall, and the confusion matrix.